# Limpieza de socioeconomico.csv

Indicadores demográficos, económicos y dotacionales de los distritos de Madrid: población, superficie, renta media, edad media, % de mayores de 65, % de extranjeros, tasa de paro, índice de vulnerabilidad, centros de salud, colegios, comisarías, parques y densidad.

Es una de las tablas base del caso 1.3 (segmentación socio-sanitaria): de aquí sale el perfil social de cada distrito.

El CSV trae 24 filas, pero Madrid tiene 21 distritos: hay distritos repetidos con otra escritura. Al final guardamos `socioeconomico_limpio.csv` con 21 filas, una por distrito.

El CSV original no se toca: la limpieza se hace en el notebook y el resultado se guarda aparte, así siempre se puede repetir desde cero.

In [1]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

# El CSV original debe estar junto al notebook o en la carpeta data del repo.
ruta = Path('socioeconomico.csv')
if not ruta.exists():
    ruta = Path('../data/socioeconomico.csv')
soc = pd.read_csv(ruta)
soc.head()

,distrito,poblacion_2024,superficie_km2,renta_media_hogar,edad_media,pct_mayores_65,pct_extranjeros,tasa_paro,indice_vulnerabilidad,num_centros_salud,num_colegios,num_comisarias,parques_m2_habitante,densidad_hab_km2
0,centro,149718,5.23,42831.0,44.2,22.100000,21.5,8.2,0.35,8,25,3,3.2,28626.8
1,ARGANZUELA,155115,6.47,39876.0,42.8,18.500000,18.2,9.5,0.42,6,22,2,8.5,23974.5
2,RETIRO,119915,5.46,NaN,47.3,25.300000,12.3,6.8,0.22,5,18,2,25.6,21962.5
3,SALAMANCA,147707,5.41,71432.0,48.1,26.800000,11.5,5.2,0.15,6,20,2,4.8,27302.6
4,CHAMARTIN,146512,9.19,62145.0,46.5,-4.900562,10.8,5.8,0.18,7,22,2,6.2,15942.5


## Revisión inicial

Miramos columnas, tipos, nulos y los nombres de distrito antes de tocar nada. Así vemos qué problemas hay y al final podemos comprobar qué ha cambiado.

In [2]:
print(soc.shape)   # 24 filas, pero hay 21 distritos
print()
print(soc.dtypes)  # num_centros_salud sale como texto: hay valores no numéricos
print()
print(soc.isna().sum())
print()
print(sorted(soc['distrito'].tolist()))

(24, 14)

distrito                  object
poblacion_2024             int64
superficie_km2           float64
renta_media_hogar        float64
edad_media               float64
pct_mayores_65           float64
pct_extranjeros          float64
tasa_paro                float64
indice_vulnerabilidad    float64
num_centros_salud         object
num_colegios               int64
num_comisarias             int64
parques_m2_habitante     float64
densidad_hab_km2         float64
dtype: object

distrito                 0
poblacion_2024           0
superficie_km2           0
renta_media_hogar        2
edad_media               0
pct_mayores_65           0
pct_extranjeros          0
tasa_paro                1
indice_vulnerabilidad    2
num_centros_salud        2
num_colegios             0
num_comisarias           0
parques_m2_habitante     0
densidad_hab_km2         0
dtype: int64

['ARGANZUELA', 'BARAJAS', 'CARABANCHEL', 'CHAMARTIN', 'CHAMBERI', 'CIUDAD LINEAL', 'Centro', 'FUENCARRAL', 'HORTALEZA', '

## Nombres de distrito

Los nombres vienen escritos de varias formas (`centro`, `Centro`, `FUENCARRAL`, `MONCLOA`, `San Blas`...). Si los dejamos así, el mismo distrito cuenta como varios.

Los normalizamos con una regla: mayúsculas, sin tildes y sin espacios sobrantes. Si el resultado ya es un nombre oficial, se queda igual. Si es solo el principio de un nombre oficial (por ejemplo `FUENCARRAL` → `FUENCARRAL-EL PARDO`), se completa, pero solo si encaja con uno único.

No usamos una lista de equivalencias escrita a mano porque Bluetab nos pidió evitar diccionarios cerrados: si mañana aparece un nombre nuevo, la regla no lo fuerza a uno equivocado, sino que se queda sin asignar y lo listamos para revisarlo.

In [3]:
OFICIALES = ["CENTRO", "ARGANZUELA", "RETIRO", "SALAMANCA", "CHAMARTIN", "TETUAN", "CHAMBERI",
             "FUENCARRAL-EL PARDO", "MONCLOA-ARAVACA", "LATINA", "CARABANCHEL", "USERA",
             "PUENTE DE VALLECAS", "MORATALAZ", "CIUDAD LINEAL", "HORTALEZA", "VILLAVERDE",
             "VILLA DE VALLECAS", "VICALVARO", "SAN BLAS-CANILLEJAS", "BARAJAS"]

def normalizar_distrito(nombre):
    if pd.isna(nombre):
        return None
    # Mayúsculas, sin tildes y sin espacios sobrantes.
    s = unicodedata.normalize('NFD', str(nombre).strip().upper())
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = ' '.join(s.split())
    if s in OFICIALES:
        return s
    # Si no es exacto, miramos si es el principio de un único nombre oficial.
    candidatos = [o for o in OFICIALES if o.startswith(s)]
    if len(candidatos) == 1:
        return candidatos[0]
    return None   # no reconocido: se deja y se lista abajo

soc['distrito'] = soc['distrito'].map(normalizar_distrito)
sin_asignar = soc[soc['distrito'].isna()]
print(f"Filas sin distrito reconocido: {len(sin_asignar)}")
if len(sin_asignar):
    print(sin_asignar)   # para revisarlas a mano

Filas sin distrito reconocido: 0


## Números escritos como texto

`num_centros_salud` llega como texto porque hay valores como `"ocho"` (Chamberí) o `"N/A"` (San Blas-Canillejas). Para poder calcular con la columna la pasamos a número:

- Si el valor es una palabra-número en español, la convertimos con una regla (se descompone la palabra y se suman sus partes: decena + unidad), no con una lista cerrada de equivalencias. Una lista escrita a mano se quedaría corta en cuanto apareciera un número que no estuviera apuntado.
- Si no se reconoce (como `"N/A"`), lo dejamos vacío, porque no sabemos cuál era el número y no lo inventamos. Los textos no reconocidos se listan para revisarlos.

In [4]:
UNIDADES = ['cero', 'uno', 'dos', 'tres', 'cuatro', 'cinco', 'seis', 'siete', 'ocho', 'nueve']
DEL_DIEZ_AL_QUINCE = ['diez', 'once', 'doce', 'trece', 'catorce', 'quince']
DECENAS = ['veinte', 'treinta', 'cuarenta', 'cincuenta', 'sesenta', 'setenta', 'ochenta', 'noventa']

def palabra_a_numero(s):
    # Regla para 0-99: unidad | diez-quince | dieci+unidad | veinti+unidad | decena [y unidad]
    if s in UNIDADES:
        return float(UNIDADES.index(s))
    if s in DEL_DIEZ_AL_QUINCE:
        return float(10 + DEL_DIEZ_AL_QUINCE.index(s))
    if s.startswith('dieci') and s[5:] in UNIDADES[1:]:
        return float(10 + UNIDADES.index(s[5:]))
    if s.startswith('veinti') and s[6:] in UNIDADES[1:]:
        return float(20 + UNIDADES.index(s[6:]))
    partes = s.split(' y ')
    if partes[0] in DECENAS:
        decena = 10 * (2 + DECENAS.index(partes[0]))
        if len(partes) == 1:
            return float(decena)
        if len(partes) == 2 and partes[1] in UNIDADES[1:]:
            return float(decena + UNIDADES.index(partes[1]))
    return None

no_reconocidos = set()

def a_numero(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    numero = pd.to_numeric(s, errors='coerce')
    if not pd.isna(numero):
        return numero   # se devuelve tal cual para no cambiar enteros a decimales
    numero = palabra_a_numero(s)
    if numero is None:
        no_reconocidos.add(s)   # 'N/A' y cualquier otro texto -> vacio, pero apuntado
        return np.nan
    return numero

for c in soc.columns:
    if c != 'distrito':
        soc[c] = soc[c].map(a_numero)

print('Textos no reconocidos (quedan vacíos):', sorted(no_reconocidos))
print(soc.dtypes)

Textos no reconocidos (quedan vacíos): []
distrito                  object
poblacion_2024             int64
superficie_km2           float64
renta_media_hogar        float64
edad_media               float64
pct_mayores_65           float64
pct_extranjeros          float64
tasa_paro                float64
indice_vulnerabilidad    float64
num_centros_salud        float64
num_colegios               int64
num_comisarias             int64
parques_m2_habitante     float64
densidad_hab_km2         float64
dtype: object


## Distritos repetidos

Tres distritos salen dos veces: Centro, Tetuán y Salamanca.

- Las copias de Salamanca son idénticas.
- En Tetuán una copia tiene vacíos el índice de vulnerabilidad y el número de centros de salud, y la otra los tiene (0,55 y 6): nos quedamos los valores de la copia que sí los tiene. Es el mismo distrito, no un dato inventado.
- En Centro la densidad difiere en 0,2 (28.626,8 frente a 28.627,0): es una diferencia de redondeo, así que nos quedamos el primer valor.

Juntamos cada distrito en una sola fila, y si una copia tiene un hueco se rellena con la otra. Antes comprobamos que no haya contradicciones de verdad (dos valores distintos y no nulos para lo mismo).

In [5]:
cols_num = [c for c in soc.columns if c != 'distrito']

# Distritos con dos valores distintos (no nulos) en alguna columna
conflictos = soc.groupby('distrito')[cols_num].nunique() > 1
cols_conflicto = conflictos.columns[conflictos.any()].tolist()
for c in cols_conflicto:
    dist = conflictos.index[conflictos[c]].tolist()
    for d in dist:
        print(f"Conflicto en {d} / {c}: {soc.loc[soc['distrito']==d, c].tolist()}")

# first() coge el primer valor NO nulo de cada columna: une las copias rellenando huecos
soc = soc.groupby('distrito', as_index=False).first()
print()
print(soc.shape)   # (21, 14)

Conflicto en CENTRO / densidad_hab_km2: [28626.8, 28627.0]

(21, 14)


## Valores imposibles

Hay valores que no pueden ser:

- % de mayores de 65 negativo (Chamartín: -4,9 y Ciudad Lineal: -5,0).
- Tasa de paro por encima de 100 (Moncloa-Aravaca: 118,1 y Hortaleza: 116,3).
- Población -999 en Barajas: es un valor centinela, no una población real.
- Índice de vulnerabilidad 1,5 en Carabanchel, cuando el resto de distritos está entre 0,15 y 0,82.

Para decidir qué es imposible usamos rangos simples: porcentajes entre 0 y 100, índice de vulnerabilidad entre 0 y 1, edad media entre 15 y 90, población, superficie, renta y densidad por encima de 0, y centros de salud, colegios, comisarías y parques sin valores negativos. Solo aparecen valores fuera de rango en los casos de arriba.

No hay forma de saber el valor real, así que los pasamos a vacío en vez de dejar un dato que sabemos que está mal.

Ojo: para el índice de vulnerabilidad asumimos que va de 0 a 1. Es un supuesto razonable por el resto de valores, pero hay que confirmarlo con Bluetab (queda en las preguntas pendientes).

Los dos valores de paro por encima de 100 (118,1 y 116,3) podrían ser 11,8 y 11,6 con la coma desplazada. Pero es una suposición que no podemos comprobar con ningún otro dato, así que no los corregimos: quedan vacíos y lo preguntamos a Bluetab.

In [6]:
RANGOS = {"poblacion_2024": (1, None), "superficie_km2": (0.1, None), "renta_media_hogar": (1, None),
          "edad_media": (15, 90), "pct_mayores_65": (0, 100), "pct_extranjeros": (0, 100),
          "tasa_paro": (0, 100), "indice_vulnerabilidad": (0, 1), "num_centros_salud": (0, None),
          "num_colegios": (0, None), "num_comisarias": (0, None),
          "parques_m2_habitante": (0, None), "densidad_hab_km2": (1, None)}

for col, (minimo, maximo) in RANGOS.items():
    malo = pd.Series(False, index=soc.index)
    if minimo is not None:
        malo |= soc[col] < minimo
    if maximo is not None:
        malo |= soc[col] > maximo
    if malo.any():
        print(f"{col}: {soc.loc[malo, ['distrito', col]].values.tolist()} -> vacio")
        soc.loc[malo, col] = np.nan

poblacion_2024: [['BARAJAS', -999]] -> vacio
pct_mayores_65: [['CHAMARTIN', -4.900561936336823], ['CIUDAD LINEAL', -4.984343577058437]] -> vacio
tasa_paro: [['HORTALEZA', 116.28518553797934], ['MONCLOA-ARAVACA', 118.13643808288734]] -> vacio
indice_vulnerabilidad: [['CARABANCHEL', 1.5]] -> vacio


## La población de Barajas

Barajas se ha quedado sin población (era -999). En `distritos.csv` tampoco aparece, así que la estimamos como **densidad × superficie**, dos columnas que sí tiene: 1.155,6 hab/km² × 41,28 km² ≈ 47.703 habitantes.

Es una estimación, no un dato real: queda anotada como pendiente de confirmar con Bluetab. La usamos porque sin población no se puede calcular la densidad ni comparar Barajas con el resto, y el propio dataset es internamente coherente (densidad = población / superficie en todos los demás distritos).

In [7]:
falta = soc['poblacion_2024'].isna()
soc.loc[falta, 'poblacion_2024'] = (soc.loc[falta, 'densidad_hab_km2'] * soc.loc[falta, 'superficie_km2']).round()
print(f"Poblacion estimada (densidad x superficie) para: {soc.loc[falta, 'distrito'].tolist()}")

# Comprobacion de coherencia: densidad = poblacion / superficie (2% de margen de redondeo)
ratio = (soc['poblacion_2024'] / soc['superficie_km2']) / soc['densidad_hab_km2']
raras = soc.loc[~ratio.between(0.98, 1.02), 'distrito'].tolist()
print(f"Distritos donde densidad no cuadra con poblacion/superficie: {raras}")

Poblacion estimada (densidad x superficie) para: ['BARAJAS']
Distritos donde densidad no cuadra con poblacion/superficie: []


## Lo que no teníamos y cómo lo resolvemos

Los 10 huecos de arriba no tienen otra fuente dentro del proyecto. En vez de dejarlos vacíos, los rellenamos en la sección siguiente con los distritos más parecidos. Es una suposición pendiente de confirmar con Bluetab.

## Rellenar los huecos con los distritos más parecidos

Quedan 10 huecos (renta de Retiro y Ciudad Lineal, % de mayores de 65 de Chamartín y Ciudad Lineal, paro de Moncloa-Aravaca, Hortaleza y Villa de Vallecas, vulnerabilidad de Carabanchel y Barajas, y centros de salud de San Blas-Canillejas). No hay otra fuente dentro del proyecto para ellos.

Con solo 21 distritos y 10 huecos, dejar vacíos esos valores nos quita un distrito entero del clustering (que no admite vacíos), y borrar los distritos sería peor. Así que los rellenamos con una regla: la media de los 3 distritos más parecidos que sí tienen ese dato (vecinos más cercanos, k=3).

- Para medir lo parecidos que son comparamos los indicadores sociales (renta, edad media, % de mayores de 65, % de extranjeros, paro y vulnerabilidad), pasados a la misma escala (restando la media y dividiendo por la desviación) para que la renta, que es enorme, no pese más que los porcentajes. En los centros de salud se añade la población.
- Solo se usan los indicadores que el distrito sí tiene, y nunca la propia columna que se está rellenando.
- Se redondea igual que el resto de la columna: renta y centros de salud a entero, porcentajes y paro a 1 decimal, vulnerabilidad a 2.

Esto es una suposición nuestra, no un dato real. Queda anotada como pendiente de confirmar con Bluetab, y si nos dan los valores oficiales se sustituyen.

In [8]:
INDICADORES = ['renta_media_hogar', 'edad_media', 'pct_mayores_65', 'pct_extranjeros',
               'tasa_paro', 'indice_vulnerabilidad']
DECIMALES = {'renta_media_hogar': 0, 'pct_mayores_65': 1, 'tasa_paro': 1,
             'indice_vulnerabilidad': 2, 'num_centros_salud': 0}

# Misma escala para todos los indicadores (media 0, desviacion 1)
usados = INDICADORES + ['poblacion_2024']
escala = (soc[usados] - soc[usados].mean()) / soc[usados].std()

huecos = soc.drop(columns='distrito').isna()
original = soc.copy()
for i, col in zip(*np.where(huecos.values)):
    col = huecos.columns[col]
    # Indicadores que este distrito si tiene, sin la columna que rellenamos.
    # La poblacion solo cuenta para los centros de salud.
    base = usados if col == 'num_centros_salud' else INDICADORES
    comp = [c for c in base if c != col and not pd.isna(original.loc[i, c])]
    candidatos = original.index[original[col].notna()]
    dist = ((escala.loc[candidatos, comp] - escala.loc[i, comp]) ** 2).mean(axis=1) ** 0.5
    vecinos = dist.dropna().sort_values().index[:3]
    valor = round(original.loc[vecinos, col].mean(), DECIMALES[col])
    soc.loc[i, col] = valor
    print(f"{original.loc[i, 'distrito']} - {col}: {valor}  (media de {', '.join(original.loc[vecinos, 'distrito'])})")

RETIRO - renta_media_hogar: 61604.0  (media de CHAMARTIN, SALAMANCA, CHAMBERI)
CHAMARTIN - pct_mayores_65: 25.2  (media de RETIRO, SALAMANCA, CHAMBERI)
MONCLOA-ARAVACA - tasa_paro: 7.6  (media de BARAJAS, FUENCARRAL-EL PARDO, CHAMBERI)
CARABANCHEL - indice_vulnerabilidad: 0.64  (media de LATINA, VILLAVERDE, SAN BLAS-CANILLEJAS)
CIUDAD LINEAL - renta_media_hogar: 42633.0  (media de MORATALAZ, BARAJAS, ARGANZUELA)
CIUDAD LINEAL - pct_mayores_65: 19.5  (media de MORATALAZ, BARAJAS, ARGANZUELA)
HORTALEZA - tasa_paro: 8.3  (media de FUENCARRAL-EL PARDO, ARGANZUELA, BARAJAS)
VILLA DE VALLECAS - tasa_paro: 15.5  (media de VICALVARO, PUENTE DE VALLECAS, VILLAVERDE)
SAN BLAS-CANILLEJAS - num_centros_salud: 7.0  (media de ARGANZUELA, TETUAN, LATINA)
BARAJAS - indice_vulnerabilidad: 0.35  (media de FUENCARRAL-EL PARDO, MONCLOA-ARAVACA, CIUDAD LINEAL)


### Comprobación del relleno

Miramos que los valores rellenados queden dentro de los rangos de antes (porcentajes de 0 a 100, vulnerabilidad de 0 a 1) y que parezcan razonables al lado de los distritos parecidos. La vulnerabilidad de Barajas, por ejemplo, sale baja, que cuadra con su renta alta y su paro bajo.

In [9]:
print(soc.drop(columns='distrito').isna().sum().sum(), "huecos que quedan")
print("Mayores de 65 y paro entre 0 y 100:", soc[['pct_mayores_65', 'tasa_paro']].stack().between(0, 100).all())
print("Vulnerabilidad entre 0 y 1:", soc['indice_vulnerabilidad'].between(0, 1).all())

0 huecos que quedan
Mayores de 65 y paro entre 0 y 100: True
Vulnerabilidad entre 0 y 1: True


In [10]:
print(soc.isna().sum()[soc.isna().sum() > 0])

Series([], dtype: int64)


## Estadísticos y guardado

Dejamos las filas en el orden oficial de distritos, con `distrito_id` de 1 a 21 (el mismo que en `distritos.csv`, para poder cruzar las tablas), y miramos los estadísticos finales para ver que todo tiene sentido.

In [11]:
soc = soc.set_index('distrito').reindex(OFICIALES).reset_index()
soc.insert(0, 'distrito_id', range(1, 22))
soc.to_csv('socioeconomico_limpio.csv', index=False)
print(soc.shape)   # (21, 15)
soc.describe().round(2)

(21, 15)


       poblacion_2024  superficie_km2  renta_media_hogar  edad_media  pct_mayores_65  pct_extranjeros  tasa_paro  indice_vulnerabilidad  num_centros_salud  num_colegios  num_comisarias  parques_m2_habitante  densidad_hab_km2
count           21.00           21.00              21.00       21.00           21.00            21.00      21.00                  21.00              21.00         21.00           21.00                 21.00             21.00
mean        157604.19           28.80           41709.81       42.95           18.95            16.55      10.54                   0.46               7.00         25.43            2.05                 11.97          14410.11
std          56809.83           50.08           13538.19        2.74            4.19             4.58       3.66                   0.20               2.45         10.01            0.67                 11.51           9945.07
min          47703.00            4.68           23456.00       37.20           11.20            10.5